## Обработка пропусков


In [ ]:
# Находим самый частый возрастной рейтинг для каждого programName
age_by_program = data.groupby('programName')['programAgeRestrictionName'].apply(
    lambda x: x.mode()[0] if not x.mode().empty else None
)

# Заполняем пропуски в исходном DataFrame
data['programAgeRestrictionName'] = data.apply(
    lambda row: age_by_program[row['programName']]
    if pd.isna(row['programAgeRestrictionName'])
    else row['programAgeRestrictionName'],
    axis=1
)

# Проверяем оставшиеся пропуски
print(data['programAgeRestrictionName'].isnull().sum())

69112


In [ ]:
# Заполняем по самой частой возрастной группе в programCategoryName
age_by_category = data.groupby('programCategoryName')['programAgeRestrictionName'].apply(
    lambda x: x.mode()[0] if not x.mode().empty else None
)

data['programAgeRestrictionName'] = data.apply(
    lambda row: age_by_category[row['programCategoryName']]
    if pd.isna(row['programAgeRestrictionName'])
    else row['programAgeRestrictionName'],
    axis=1
)

print(data['programAgeRestrictionName'].isnull().sum())  # Должно быть 0
print(data['programAgeRestrictionName'].value_counts())  # Распределение рейтингов

print(data[data['programAgeRestrictionName'].isna()]['programName'].unique())
print(data[data['programAgeRestrictionName'].isna()]['programCategoryName'].unique())
print(data[data['programAgeRestrictionName'].isna()]['programTypeName'].unique())

259
programAgeRestrictionName
16+    1137732
12+     284904
0+       42779
18+      31909
6+       30302
14+        143
5+          44
21+         20
3+           8
4+           3
Name: count, dtype: int64
['Вести. Церковь и мир' 'Рыбий жыр'
 'Рождественское интервью Святейшего Патриарха Кирилла'
 'К 75-летию. Большое интервью Святейшего Патриарха Московского и всея Руси Кирилла'
 'Вести. Слово' 'Схождение благодатного огня' 'Церковь и мир'
 'Новогоднее обращение Президента России Владимира Владимировича Путина'
 'Поздравление Президента РФ к Дню защитника Отечества']
['Религиозная передача' 'Стиль жизни и досуг' 'Поздравление']
['Социально-политическая программа' 'Прочее']


Для 255 записей (0.02% данных) можно заполнить самым частым рейтингом "16+"
тк это наиболее распространённый рейтинг (1 119 251 случай), поэтому он будет наименее "вредной" заменой.
+ по категориям можно заметить что эти программы как раз относятся к категории 16+

In [ ]:
data['programAgeRestrictionName'] = data['programAgeRestrictionName'].fillna('16+')

# Проверка распределения
print(data['programAgeRestrictionName'].value_counts(normalize=True))

# Проверка оставшихся пропусков
print("Осталось пропусков:", data['programAgeRestrictionName'].isna().sum())

programAgeRestrictionName
16+    0.744708
12+    0.186443
0+     0.027995
18+    0.020881
6+     0.019830
14+    0.000094
5+     0.000029
21+    0.000013
3+     0.000005
4+     0.000002
Name: proportion, dtype: float64
Осталось пропусков: 0


После добавления валидации появились новые пропуски


In [ ]:
# ДЛЯ ПРЕДСКАЗАНИЙ Заполняем все пропуски нулями
data['predictions'] = data['predictions'].fillna(0)

print(f"{data['predictions'].isnull().sum()} пропусков в predictions")

0 пропусков в predictions


In [ ]:
# Заполняем в зависимости от типа дня
day_type_avg = data.groupby('researchDayType')['SalesRtgPer'].mean()

# Посмотрим средние значения по типам дней
print("Средние значения по типам дней:")
print(day_type_avg)

# Заполняем пропуски средними по типу дня
data['SalesRtgPer'] = data.groupby('researchDayType')['SalesRtgPer'].transform(
    lambda x: x.fillna(x.mean())
)

# Для оставшихся пропусков (если есть) заполняем общим средним
data['SalesRtgPer'] = data['SalesRtgPer'].fillna(
    data['SalesRtgPer'].mean()
)

Средние значения по типам дней:
researchDayType
E    0.818327
H    0.867824
W    0.699719
Name: SalesRtgPer, dtype: float64


In [ ]:
print(data.isnull().sum())

researchDate                 0
researchDayType              0
tvCompanyId                  0
tvCompanyName                0
breaksSpotId                 0
breaksStartTime              0
breaksFinishTime             0
breaksDuration               0
breaksPrimeTimeStatusName    2
programFirstIssueDate        0
programAgeRestrictionName    0
programName                  0
programTypeName              0
programCategoryName          0
programStartTime             0
programFinishTime            0
programDuration              0
SalesRtgPer                  0
predictions                  0
hour                         0
minute                       0
second                       0
day                          0
month                        0
year                         0
dtype: int64


Тк всего 2 строчки то можно их просто удалить


In [ ]:
data = data.dropna(subset=['breaksPrimeTimeStatusName'])